In [1]:
import os
import shutil
import sys

from langgraph.graph import StateGraph,START,END
from pydantic import BaseModel, Field

from typing import List, Dict, Any, Optional

In [2]:

class Task(BaseModel):
    filename: str = Field(..., description="The name of the file associated with the task")
    file_path: str = Field(..., description="The path to the file associated with the task")
    description: str = Field(..., description="A detailed description of the task to be performed")

class State(BaseModel):
    topic: str = Field(..., description="user prompt to create website application")
    plan: str = Field(..., description="a detailed plan to create the website application")
    tasks: List[Task]
    

In [19]:
# Defining the LLM
from langchain_aws import ChatBedrockConverse
from dotenv import load_dotenv
load_dotenv()

LLM_MODEL_ID = "us.meta.llama3-3-70b-instruct-v1:0"
LLM_CODER_MODEL_ID = "arn:aws:bedrock:us-east-1:868522398119:inference-profile/us.meta.llama3-2-1b-instruct-v1:0"
LLM_CODER_MODEL_ID2="anthropic.claude-3-haiku-20240307-v1:0"
LLM_REGION = "us-east-1"
llm=ChatBedrockConverse(
    model=LLM_MODEL_ID,
    region_name=LLM_REGION,
)


In [33]:
from langchain_core.messages import SystemMessage
import json
def orchastrator(state:State)->State:
    plan_prompt = f"Given the topic '{state.topic}', create a detailed plan to create a website application."
    state.plan = llm.invoke([SystemMessage(content=plan_prompt)]).content
    
    tasks_prompt = f"Based on the following plan, generate a list of tasks with filenames, file_paths, and descriptions in JSON format:\n\n{state.plan}\n\nReturn only the JSON array."
    tasks_response = llm.invoke([SystemMessage(content=tasks_prompt)]).content
    tasks_data = json.loads(tasks_response)
    state.tasks = [Task(**task) for task in tasks_data]
    
    return state

In [31]:
graph=StateGraph(State)
graph.add_node("orchastrator", orchastrator)
graph.add_edge(START, "orchastrator")
graph.add_edge("orchastrator", END)


graph=graph.compile()


In [35]:

temp_task=Task(filename="index.html", file_path="/path/to/index.html", description="Create the main HTML file for the bakery website.")
state=State(topic="Create a website application for a local bakery that includes features such as an online menu, ordering system, and contact information.", plan="", tasks=[temp_task])
graph.invoke(state)

/home/vashuthegreat/Projects/Coder-Buddy-All-in-One-Agent-/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='plan', input_value=AIMessage(content="**Loca...': 0, 'cache_read': 0}}), input_type=AIMessage])
  PydanticSerializationUnexpectedValue(Expected `list[Task]` - serialized value may not be as expected [field_name='tasks', input_value=AIMessage(content="Based ...': 0, 'cache_read': 0}}), input_type=AIMessage])
  return self.__pydantic_serializer__.to_python(


{'topic': 'Create a website application for a local bakery that includes features such as an online menu, ordering system, and contact information.',
 'plan': AIMessage(content="**Local Bakery Website Application Plan**\n\n**Introduction:**\nThe goal of this project is to create a website application for a local bakery that showcases their menu, allows customers to place orders online, and provides contact information. The website will be user-friendly, visually appealing, and easy to navigate.\n\n**Features:**\n\n1. **Online Menu:**\n\t* Display a list of available baked goods, including cakes, pastries, bread, and other sweet and savory treats.\n\t* Include high-quality images of each item to showcase the bakery's products.\n\t* Provide detailed descriptions of each item, including ingredients, prices, and nutritional information.\n2. **Ordering System:**\n\t* Allow customers to place orders online for pickup or delivery.\n\t* Implement a secure payment gateway to process transaction